In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

df = pd.read_csv('../data/GlobalWeatherRepository.csv')
df['last_updated'] = pd.to_datetime(df['last_updated'])
df = df.sort_values('last_updated').reset_index(drop=True)

print(f"✅ Loaded and sorted: {df.shape}")
print(f"Date range: {df['last_updated'].min()} → {df['last_updated'].max()}")

✅ Loaded and sorted: (145212, 41)
Date range: 2024-05-16 01:45:00 → 2026-06-03 19:00:00


In [3]:
air_quality_cols = [
    'air_quality_Carbon_Monoxide', 'air_quality_Ozone',
    'air_quality_Nitrogen_dioxide', 'air_quality_Sulphur_dioxide',
    'air_quality_PM2.5', 'air_quality_PM10'
]

before = df[air_quality_cols].lt(0).sum()
for col in air_quality_cols:
    df[col] = df[col].where(df[col] >= 0, np.nan)
after = df[air_quality_cols].isnull().sum()

print("Sentinel values (negatives / -9999) replaced with NaN:")
print(pd.DataFrame({'negative_count_before': before, 'nan_count_after': after}))
print(f"\nTotal NaNs introduced: {after.sum():,}")

Sentinel values (negatives / -9999) replaced with NaN:
                              negative_count_before  nan_count_after
air_quality_Carbon_Monoxide                       1                1
air_quality_Ozone                                 0                0
air_quality_Nitrogen_dioxide                      0                0
air_quality_Sulphur_dioxide                       1                1
air_quality_PM2.5                                 0                0
air_quality_PM10                                  2                2

Total NaNs introduced: 4


In [4]:
# Records before clipping
wind_out    = (df['wind_kph'] > 200).sum()
pressure_out = ((df['pressure_mb'] < 870) | (df['pressure_mb'] > 1084)).sum()

df['wind_kph']    = df['wind_kph'].clip(upper=200)
df['gust_kph']    = df['gust_kph'].clip(upper=250)
df['wind_mph']    = df['wind_mph'].clip(upper=130)
df['gust_mph']    = df['gust_mph'].clip(upper=155)
df['pressure_mb'] = df['pressure_mb'].clip(870, 1084)

print(f"wind_kph rows clipped     : {wind_out}")
print(f"pressure_mb rows clipped  : {pressure_out}")
print(f"\nNew wind_kph max      : {df['wind_kph'].max()}")
print(f"New pressure_mb range : {df['pressure_mb'].min()} – {df['pressure_mb'].max()}")
print("✅ Physical outliers capped")

wind_kph rows clipped     : 4
pressure_mb rows clipped  : 2

New wind_kph max      : 200.0
New pressure_mb range : 947.0 – 1084.0
✅ Physical outliers capped


In [5]:
continent_map = {
    'Afghanistan': 'Asia', 'Albania': 'Europe', 'Algeria': 'Africa', 'Andorra': 'Europe',
    'Angola': 'Africa', 'Antigua and Barbuda': 'North America', 'Argentina': 'South America',
    'Armenia': 'Asia', 'Australia': 'Oceania', 'Austria': 'Europe', 'Azerbaijan': 'Asia',
    'Bahamas': 'North America', 'Bahrain': 'Asia', 'Bangladesh': 'Asia', 'Barbados': 'North America',
    'Belarus': 'Europe', 'Belgium': 'Europe', 'Belize': 'North America', 'Benin': 'Africa',
    'Bhutan': 'Asia', 'Bolivia': 'South America', 'Bosnia and Herzegovina': 'Europe',
    'Botswana': 'Africa', 'Brazil': 'South America', 'Brunei': 'Asia', 'Bulgaria': 'Europe',
    'Burkina Faso': 'Africa', 'Burundi': 'Africa', 'Cambodia': 'Asia', 'Cameroon': 'Africa',
    'Canada': 'North America', 'Cape Verde': 'Africa', 'Central African Republic': 'Africa',
    'Chad': 'Africa', 'Chile': 'South America', 'China': 'Asia', 'Colombia': 'South America',
    'Comoros': 'Africa', 'Congo': 'Africa', 'Costa Rica': 'North America', 'Croatia': 'Europe',
    'Cuba': 'North America', 'Cyprus': 'Asia', 'Czech Republic': 'Europe', 'Czechia': 'Europe',
    'Denmark': 'Europe', 'Djibouti': 'Africa', 'Dominican Republic': 'North America',
    'Ecuador': 'South America', 'Egypt': 'Africa', 'El Salvador': 'North America',
    'Equatorial Guinea': 'Africa', 'Eritrea': 'Africa', 'Estonia': 'Europe', 'Eswatini': 'Africa',
    'Ethiopia': 'Africa', 'Fiji': 'Oceania', 'Finland': 'Europe', 'France': 'Europe',
    'Gabon': 'Africa', 'Gambia': 'Africa', 'Georgia': 'Asia', 'Germany': 'Europe',
    'Ghana': 'Africa', 'Greece': 'Europe', 'Guatemala': 'North America', 'Guinea': 'Africa',
    'Guinea-Bissau': 'Africa', 'Guyana': 'South America', 'Haiti': 'North America',
    'Honduras': 'North America', 'Hungary': 'Europe', 'Iceland': 'Europe', 'India': 'Asia',
    'Indonesia': 'Asia', 'Iran': 'Asia', 'Iraq': 'Asia', 'Ireland': 'Europe', 'Israel': 'Asia',
    'Italy': 'Europe', 'Jamaica': 'North America', 'Japan': 'Asia', 'Jordan': 'Asia',
    'Kazakhstan': 'Asia', 'Kenya': 'Africa', 'Kosovo': 'Europe', 'Kuwait': 'Asia',
    'Kyrgyzstan': 'Asia', 'Laos': 'Asia', 'Latvia': 'Europe', 'Lebanon': 'Asia',
    'Lesotho': 'Africa', 'Liberia': 'Africa', 'Libya': 'Africa', 'Liechtenstein': 'Europe',
    'Lithuania': 'Europe', 'Luxembourg': 'Europe', 'Madagascar': 'Africa', 'Malawi': 'Africa',
    'Malaysia': 'Asia', 'Maldives': 'Asia', 'Mali': 'Africa', 'Malta': 'Europe',
    'Mauritania': 'Africa', 'Mauritius': 'Africa', 'Mexico': 'North America', 'Moldova': 'Europe',
    'Monaco': 'Europe', 'Mongolia': 'Asia', 'Montenegro': 'Europe', 'Morocco': 'Africa',
    'Mozambique': 'Africa', 'Myanmar': 'Asia', 'Namibia': 'Africa', 'Nepal': 'Asia',
    'Netherlands': 'Europe', 'New Zealand': 'Oceania', 'Nicaragua': 'North America',
    'Niger': 'Africa', 'Nigeria': 'Africa', 'North Korea': 'Asia', 'North Macedonia': 'Europe',
    'Norway': 'Europe', 'Oman': 'Asia', 'Pakistan': 'Asia', 'Panama': 'North America',
    'Papua New Guinea': 'Oceania', 'Paraguay': 'South America', 'Peru': 'South America',
    'Philippines': 'Asia', 'Poland': 'Europe', 'Portugal': 'Europe', 'Qatar': 'Asia',
    'Romania': 'Europe', 'Russia': 'Europe', 'Rwanda': 'Africa', 'Saudi Arabia': 'Asia',
    'Senegal': 'Africa', 'Serbia': 'Europe', 'Sierra Leone': 'Africa', 'Singapore': 'Asia',
    'Slovakia': 'Europe', 'Slovenia': 'Europe', 'Solomon Islands': 'Oceania', 'Somalia': 'Africa',
    'South Africa': 'Africa', 'South Korea': 'Asia', 'South Sudan': 'Africa', 'Spain': 'Europe',
    'Sri Lanka': 'Asia', 'Sudan': 'Africa', 'Suriname': 'South America', 'Sweden': 'Europe',
    'Switzerland': 'Europe', 'Syria': 'Asia', 'Taiwan': 'Asia', 'Tajikistan': 'Asia',
    'Tanzania': 'Africa', 'Thailand': 'Asia', 'Timor-Leste': 'Asia', 'Togo': 'Africa',
    'Trinidad and Tobago': 'North America', 'Tunisia': 'Africa', 'Turkey': 'Asia',
    'Turkmenistan': 'Asia', 'Uganda': 'Africa', 'Ukraine': 'Europe',
    'United Arab Emirates': 'Asia', 'United Kingdom': 'Europe',
    'United States of America': 'North America', 'United States': 'North America',
    'Uruguay': 'South America', 'Uzbekistan': 'Asia', 'Vatican City': 'Europe',
    'Venezuela': 'South America', 'Vietnam': 'Asia', 'Yemen': 'Asia',
    'Zambia': 'Africa', 'Zimbabwe': 'Africa',
    'Democratic Republic of the Congo': 'Africa', "Côte d'Ivoire": 'Africa',
    'Sao Tome and Principe': 'Africa', 'Palestine': 'Asia',
}
month_to_season = {
    12: 'Winter', 1: 'Winter',  2: 'Winter',
     3: 'Spring', 4: 'Spring',  5: 'Spring',
     6: 'Summer', 7: 'Summer',  8: 'Summer',
     9: 'Autumn', 10: 'Autumn', 11: 'Autumn',
}

df['continent'] = df['country'].map(continent_map).fillna('Other')
df['season']    = df['last_updated'].dt.month.map(month_to_season)

print("Continent breakdown:")
print(df['continent'].value_counts())
print(f"\nSeason breakdown:")
print(df['season'].value_counts())
print("✅ Continent and season columns added")

Continent breakdown:
continent
Asia             36180
Africa           35487
Europe           32372
Other            15932
North America    13346
South America     8912
Oceania           2983
Name: count, dtype: int64

Season breakdown:
season
Spring    38977
Summer    35949
Autumn    35435
Winter    34851
Name: count, dtype: int64
✅ Continent and season columns added


In [6]:
df['year']        = df['last_updated'].dt.year
df['month']       = df['last_updated'].dt.month
df['day_of_week'] = df['last_updated'].dt.dayofweek
df['day_of_year'] = df['last_updated'].dt.dayofyear
df['hour']        = df['last_updated'].dt.hour

print("Datetime features added:")
print(df[['last_updated', 'year', 'month', 'day_of_week', 'day_of_year', 'hour']].head(5))
print("✅ Datetime feature columns done")

Datetime features added:
         last_updated  year  month  day_of_week  day_of_year  hour
0 2024-05-16 01:45:00  2024      5            3          137     1
1 2024-05-16 02:45:00  2024      5            3          137     2
2 2024-05-16 02:45:00  2024      5            3          137     2
3 2024-05-16 02:45:00  2024      5            3          137     2
4 2024-05-16 02:45:00  2024      5            3          137     2
✅ Datetime feature columns done


In [7]:
def flag_outliers_iqr(series):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    return (series < Q1 - 1.5 * IQR) | (series > Q3 + 1.5 * IQR)

flag_cols = ['temperature_celsius', 'precip_mm', 'wind_kph', 'humidity', 'pressure_mb']

for col in flag_cols:
    df[f'{col}_outlier'] = flag_outliers_iqr(df[col])

summary = df[[f'{c}_outlier' for c in flag_cols]].sum()
print("Outlier flag counts (IQR method):")
print(summary)
print(f"\nRows flagged in at least one column: {df[[f'{c}_outlier' for c in flag_cols]].any(axis=1).sum():,}")
print("✅ Outlier flags added (rows kept — flagged only)")

Outlier flag counts (IQR method):
temperature_celsius_outlier     2407
precip_mm_outlier              29361
wind_kph_outlier                2453
humidity_outlier                   0
pressure_mb_outlier             4186
dtype: int64

Rows flagged in at least one column: 36,023
✅ Outlier flags added (rows kept — flagged only)


In [8]:
# Sort per city before computing lags
df = df.sort_values(['location_name', 'last_updated']).reset_index(drop=True)

df['temp_lag_1']          = df.groupby('location_name')['temperature_celsius'].shift(1)
df['temp_lag_7']          = df.groupby('location_name')['temperature_celsius'].shift(7)
df['temp_rolling_7_mean'] = df.groupby('location_name')['temperature_celsius'] \
                               .transform(lambda x: x.rolling(7, min_periods=1).mean())
df['temp_rolling_7_std']  = df.groupby('location_name')['temperature_celsius'] \
                               .transform(lambda x: x.rolling(7, min_periods=1).std().fillna(0))

print("Lag + rolling features added:")
print(df[['location_name', 'last_updated', 'temperature_celsius',
          'temp_lag_1', 'temp_lag_7',
          'temp_rolling_7_mean', 'temp_rolling_7_std']].head(10))
print(f"\nNaN from lag_1 : {df['temp_lag_1'].isnull().sum():,}")
print(f"NaN from lag_7 : {df['temp_lag_7'].isnull().sum():,}")

Lag + rolling features added:
       location_name        last_updated  temperature_celsius  temp_lag_1  \
0  'S Gravenjansdijk 2024-05-31 16:15:00                16.00         NaN   
1  'S Gravenjansdijk 2024-06-01 16:30:00                16.00       16.00   
2  'S Gravenjansdijk 2024-06-04 16:15:00                18.00       16.00   
3  'S Gravenjansdijk 2024-06-05 16:15:00                15.00       18.00   
4  'S Gravenjansdijk 2024-06-11 16:15:00                15.20       15.00   
5  'S Gravenjansdijk 2024-06-14 16:00:00                16.50       15.20   
6  'S Gravenjansdijk 2024-06-15 16:00:00                17.40       16.50   
7  'S Gravenjansdijk 2024-06-16 15:45:00                16.10       17.40   
8  'S Gravenjansdijk 2024-06-20 15:45:00                21.30       16.10   
9  'S Gravenjansdijk 2024-06-21 15:45:00                16.40       21.30   

   temp_lag_7  temp_rolling_7_mean  temp_rolling_7_std  
0         NaN                16.00                0.00  
1       

In [9]:
print(f"Final dataframe shape: {df.shape}")
print(f"\nNew columns added:")
new_cols = ['continent', 'season', 'year', 'month', 'day_of_week', 'day_of_year', 'hour',
            'temp_lag_1', 'temp_lag_7', 'temp_rolling_7_mean', 'temp_rolling_7_std'] + \
           [f'{c}_outlier' for c in ['temperature_celsius','precip_mm','wind_kph','humidity','pressure_mb']]
print(new_cols)

print(f"\nRemaining NaNs (air quality only — expected):")
print(df.isnull().sum()[df.isnull().sum() > 0])

df.to_csv('../data/df_clean.csv', index=False)
print(f"\n✅ Saved: data/df_clean.csv  ({df.shape[0]:,} rows × {df.shape[1]} cols)")

Final dataframe shape: (145212, 57)

New columns added:
['continent', 'season', 'year', 'month', 'day_of_week', 'day_of_year', 'hour', 'temp_lag_1', 'temp_lag_7', 'temp_rolling_7_mean', 'temp_rolling_7_std', 'temperature_celsius_outlier', 'precip_mm_outlier', 'wind_kph_outlier', 'humidity_outlier', 'pressure_mb_outlier']

Remaining NaNs (air quality only — expected):
air_quality_Carbon_Monoxide       1
air_quality_Sulphur_dioxide       1
air_quality_PM10                  2
temp_lag_1                      257
temp_lag_7                     1601
dtype: int64

✅ Saved: data/df_clean.csv  (145,212 rows × 57 cols)
